[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/33_beam_search_solution.ipynb)

# ✅ Solution: Beam Search Decoding

Implement **beam search** — the classic decoding algorithm for sequence generation.

### Signature
```python
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token) -> list[int]:
    # log_prob_fn: takes token list, returns (V,) log-probabilities
    # Returns: best sequence (list of ints)
```

### Algorithm
1. Start with `[(0.0, [start_token])]`
2. Each step: expand each beam with top-k next tokens
3. Keep top `beam_width` beams by total log-probability
4. Stop when best beam ends with `eos_token` or `max_len` reached


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import math


In [ ]:
# ✅ SOLUTION

import jax.numpy as jnp
def beam_search(log_prob_fn,start_token,max_len,beam_width,eos_token):
    beams=[(0.,[start_token])]; done=[]
    for _ in range(max_len):
        candidates=[]
        for score,seq in beams:
            if seq[-1]==eos_token: done.append((score,seq)); continue
            lp=log_prob_fn(jnp.array(seq))
            for token in jnp.argsort(lp)[-beam_width:]: candidates.append((score+float(lp[token]),seq+[int(token)]))
        if not candidates: break
        beams=sorted(candidates,reverse=True,key=lambda x:x[0])[:beam_width]
        if all(s[-1]==eos_token for _,s in beams): done.extend(beams); break
    return sorted(done+beams,reverse=True,key=lambda x:x[0])[0][1]


In [ ]:
# Verify
print(beam_search)


In [ ]:
from jax_judge import check
check("beam_search")
